
# 04D1E_v3 — LANDFIRE Recovery / Resume

**Build ID: `04D1E_V3_LANDFIRE_RECOVERY_RESUME`**

04D1C V5 is frozen.

04D1D V4 passed the resolved-vintage production pilot with:

- all three resolved vintages tested;
- all 15 pilot product tiles passing raw raster QA;
- physical native-cell alignment across products;
- CH/CBH/CBD cell-center alignment;
- one feature row per pilot unit;
- exact frozen-V5 schema matching.

This notebook scales the same production implementation to **every currently
resolved LANDFIRE block**:

- LF2016R — 2020 fires
- LF2022 — 2023 fires
- LF2023 — 2024 fires

LF2019L and LF2020 remain excluded until archived raw sources are acquired.

## Production principles

1. **Frozen 04D1C V5 feature contract is immutable.**
2. **No per-unit REST sampling.**
3. One cached raw 30 m raster tile per `vintage × block × product`.
4. Native-cell sampling and aggregation are local.
5. Each block writes a restartable Parquet feature shard.
6. Successful blocks are skipped on rerun after shard/schema validation.
7. Failed blocks are logged and production continues.
8. Final consolidation occurs only after all expected blocks pass.
9. Missing positive-conditioned canopy statistics remain missing by design;
   they are not imputed in the extraction stage.


The full currently resolved production universe is expected to contain
**974 version-block combinations**. The manifest cell recomputes the exact
counts from the frozen unit index at runtime rather than relying on this
documentation value.


## Recovery behavior

This notebook is **not a clean rerun**.

It deliberately reuses the existing production locations:

- `data/processed/landfire_resolved_vintage_full_production_v1/feature_shards`
- `data/raw/landfire/resolved_vintage_block_cache_v3`

Every existing shard is checked first. A valid shard is skipped.

Only a block whose shard is missing or invalid is recomputed.

### V2 fixes

1. **Directionless-block positional bug**

   When a block has no reliable directional samples, V1 could create an empty
   positional DataFrame with no `unit_uid` column and then call
   `.groupby("unit_uid")`, causing `KeyError: 'unit_uid'`.

   V2 explicitly skips an empty directional positional region.

2. **Production-summary bookkeeping bug**

   Computed results carried the lists `missing_v5_features` and
   `unexpected_landfire_features`; resumed results carried count fields.
   V2 normalizes both representations before summary construction.

3. **Transient ArcGIS output delivery**

   Temporary `_ags_*.tif` URLs are retried at the product-download level before
   escalating to a whole-block retry. Files are written atomically through a
   `.part` file.

4. **Fast resume audit**

   Existing shards are validated from Parquet schema plus only the minimal
   identity columns, instead of reading all ~600 feature columns just to decide
   whether a block can be skipped.


## V3 correction — adaptive request-aware tile extent

The fixed `10 km core + 360 m halo` tile extent is sufficient for the
**directional offset from a line sample**, but it does not guarantee coverage
of every request when the suppression unit itself extends away from the block
reference location before the ±300 m context offset is applied.

V2 therefore exposed deterministic failures such as:

`FBFM40: 2 requests fell outside cached tile.`

V3 changes only the **download/cache extent**, not the feature geometry.

For each production block:

1. build the complete frozen-V5 context request table first;
2. take the actual min/max request `x_m/y_m`;
3. add a 60 m safety pad (two native 30 m cells);
4. snap that bbox to the service's native grid;
5. reuse an existing cache tile only if its raster bounds fully cover that
   snapped required bbox;
6. otherwise refresh that one product tile with the larger required extent.

Thus the feature locations remain exactly the same as frozen V5. Only the
cached raster envelope becomes request-aware.


## 0. Configuration

## Inherited 04D1D V2 contract — canonical 04A merge

The terrain operational unit index may already contain approach-direction
columns inherited from the terrain stage.

V1 merged the canonical 04A table without removing those overlapping fields,
so pandas produced suffixed columns such as:

- `approach_direction_reliable_x`
- `approach_direction_reliable_y`

and the expected unsuffixed field disappeared.

V2 now:

1. detects overlapping 04A columns in the terrain unit index;
2. drops only those duplicate copies;
3. merges the canonical 04A table;
4. fails if any `_x` / `_y` merge artifacts remain;
5. verifies every canonical 04A field exists.

No cohort, direction estimate, LANDFIRE feature, or production geometry changes.


## Inherited 04D1D V3 contract — physical native-cell alignment

V2 required all five product tile transforms to be identical.

That requirement is unnecessarily strict. Independently exported product tiles
may have different local bounds/origins while still representing the same
underlying 30 m cell lattice at every requested location. In that case local
`row/col` indices differ even though the selected physical cell centers agree.

V3 therefore freezes the correct cross-product contract:

### Individual products

Each product is validated and sampled on its own returned native 30 m transform.

### Joint CH/CBH/CBD features

For every context request:

1. derive the physical center coordinate of the selected CH cell;
2. derive the physical center coordinate of the selected CBH cell;
3. derive the physical center coordinate of the selected CBD cell;
4. require those centers to agree within `1e-6 m`;
5. join/collapse joint canopy by physical cell-center coordinate rather than
   local tile row/column.

If CH/CBH/CBD physical centers do not align, the production still **hard fails**.
No silent resampling or cross-cell canopy combination is allowed.

FBFM40 and CC are not required to share the same local tile transform because
they are never combined cell-by-cell with the canopy-derived variables.
Their cross-product center offsets are retained as QA diagnostics.


## Inherited 04D1D V4 contract — exact frozen-V5 feature emission

The V3 production mechanics passed. Its remaining failure was purely a schema
emission mismatch.

### V3 diagnosis

Five frozen features were missing:

- `landfire_line_fbfm_group_entropy`
- `landfire_fireward_near_fbfm_group_entropy`
- `landfire_fireward_far_fbfm_group_entropy`
- `landfire_protected_near_fbfm_group_entropy`
- `landfire_protected_far_fbfm_group_entropy`

The naming helper incorrectly treated the statistic key `group_entropy` as if
it were a broad-group fraction and emitted:

`landfire_<region>_fbfm_group_entropy_fraction`

instead.

The other unexpected fields were internal canopy statistics intentionally
computed by the aggregation helper but not included in the frozen V5 feature
dictionary.

### V4 contract

1. `group_entropy` is explicitly mapped to
   `landfire_<region>_fbfm_group_entropy`.
2. Region-level aggregation may compute extra internal statistics, but a field
   is emitted only if its exact name occurs in
   `LANDFIRE_FEATURE_DICTIONARY_V5.csv`.
3. The frozen 04D1C V5 dictionary remains unchanged.
4. The validated V3 raw tile cache is reused; no redownload is required.

This is a production-schema correction only.


In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "04D1E_V3_LANDFIRE_RECOVERY_RESUME"

from pathlib import Path
import sys, subprocess, importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

APPROACH_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "fireline_extended_exact_approach_v3" /
    "extended_exact_approach_features_v3.parquet"
)

TERRAIN_UNIT_INDEX_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "terrain_3dep_production_v1" /
    "terrain_operational_unit_index_v1.parquet"
)

TERRAIN_BLOCK_SAMPLE_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "terrain_3dep_production_v1" /
    "block_samples"
)

FEATURE_DICTIONARY_PATH = (
    PROJECT_ROOT / "outputs" /
    "landfire_feature_architecture_v5" /
    "LANDFIRE_FEATURE_DICTIONARY_V5.csv"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "landfire_resolved_vintage_full_production_v1"
)

PROCESSED_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "landfire_resolved_vintage_full_production_v1"
)

CACHE_ROOT = (
    PROJECT_ROOT / "data" / "raw" /
    "landfire" /
    "resolved_vintage_block_cache_v3"
)

SHARD_ROOT = (
    PROCESSED_ROOT / "feature_shards"
)

for p in [
    OUTPUT_ROOT,
    PROCESSED_ROOT,
    CACHE_ROOT,
    SHARD_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

for p in [
    APPROACH_PATH,
    TERRAIN_UNIT_INDEX_PATH,
    TERRAIN_BLOCK_SAMPLE_ROOT,
    FEATURE_DICTIONARY_PATH,
]:
    assert p.exists(), p

VERSION_BY_FIRE_YEAR = {
    2020: "LF2016R",
    2021: "LF2019L",
    2022: "LF2020",
    2023: "LF2022",
    2024: "LF2023",
}

RESOLVED_VERSIONS = [
    "LF2016R",
    "LF2022",
    "LF2023",
]

PRODUCTS = [
    "FBFM40",
    "CC",
    "CH",
    "CBH",
    "CBD",
]

REGION_DEPTHS_M = {
    "line": [0.0],
    "fireward_near": [-30.0, -60.0, -90.0],
    "fireward_far": [-150.0, -210.0, -300.0],
    "protected_near": [30.0, 60.0, 90.0],
    "protected_far": [150.0, 210.0, 300.0],
}

DIRECTIONAL_REGIONS = {
    "fireward_near",
    "fireward_far",
    "protected_near",
    "protected_far",
}

BLOCK_SIZE_M = 10_000.0
BLOCK_HALO_M = 360.0

# V3 production download safety:
# context requests already contain the ±300 m directional geometry, so the
# tile only needs a small additional native-grid safety pad around their
# observed min/max extent.
REQUEST_BBOX_PAD_M = 60.0

# ------------------------------------------------------------------
# Full-production controls.
# ------------------------------------------------------------------

# Keep True for normal production. Existing validated raw block tiles are
# reused, including the 04D1D pilot cache.
USE_TILE_CACHE = True

# Resume completed feature shards after validating unit count + frozen schema.
RESUME_COMPLETED_BLOCKS = True

# Continue after a block failure and record it for targeted rerun.
CONTINUE_ON_BLOCK_ERROR = True

# Number of attempts for transient web/download failures.
MAX_BLOCK_ATTEMPTS = 3
RETRY_BACKOFF_S = 15

REQUEST_TIMEOUT_S = 180

# ArcGIS exportImage can return an href before the temporary TIFF is
# immediately retrievable. Retry that href locally before a whole-block retry.
HREF_DOWNLOAD_ATTEMPTS = 4
HREF_RETRY_SLEEP_S = 2.0
EXPORT_REISSUE_ATTEMPTS = 2

# Optional development limit. Keep None for full production.
MAX_BLOCKS_TOTAL = None

# Optional version subset, e.g. ["LF2023"]. Keep all three for full run.
PRODUCTION_VERSIONS = [
    "LF2016R",
    "LF2022",
    "LF2023",
]

MOSAIC_RULE = {
    "mosaicMethod": "esriMosaicNorthwest",
    "mosaicOperation": "MT_FIRST",
    "ascending": True,
}

SERVICE_URLS = {
    "LF2016R": {
        p: (
            "https://lfps.usgs.gov/arcgis/rest/services/"
            f"Landfire_LF2016/LF2016_{p}_CONUS/ImageServer"
        )
        for p in PRODUCTS
    },
    "LF2022": {
        p: (
            "https://lfps.usgs.gov/arcgis/rest/services/"
            f"Landfire_LF2022/LF2022_{p}_CONUS/ImageServer"
        )
        for p in PRODUCTS
    },
    "LF2023": {
        p: (
            "https://lfps.usgs.gov/arcgis/rest/services/"
            f"Landfire_LF2023/LF2023_{p}_CONUS/ImageServer"
        )
        for p in PRODUCTS
    },
}

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("Production versions:", PRODUCTION_VERSIONS)
print("Resume completed blocks:", RESUME_COMPLETED_BLOCKS)
print("Continue on block error:", CONTINUE_ON_BLOCK_ERROR)
print("Max blocks total:", MAX_BLOCKS_TOTAL)

## 1. Imports

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "requests": "requests",
    "rasterio": "rasterio",
    "pyarrow": "pyarrow",
    "tqdm": "tqdm",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package]
        )

from datetime import datetime, timezone
import hashlib
import traceback
import json
import math
import time

import numpy as np
import pandas as pd
import requests
import rasterio
import pyarrow.parquet as pq

from rasterio.crs import CRS
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 360)

session = requests.Session()
session.headers.update({
    "User-Agent": "wildfire-fireline-effectiveness/04D1D"
})

## 2. Load frozen unit universe and V5 feature contract

In [ ]:
unit_index = pd.read_parquet(TERRAIN_UNIT_INDEX_PATH)
approach = pd.read_parquet(APPROACH_PATH)
feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)

FROZEN_V5_FEATURE_SET = set(
    feature_dictionary["feature"].astype(str)
)

assert len(FROZEN_V5_FEATURE_SET) == len(feature_dictionary)

assert unit_index["unit_uid"].is_unique
assert approach["unit_uid"].is_unique
assert feature_dictionary["feature"].is_unique

required_unit_cols = [
    "unit_uid",
    "_irwin_norm",
    "IncidentName",
    "Year",
    "_engagement_norm",
    "terrain_block_id",
    "terrain_block_ix",
    "terrain_block_iy",
]

missing = [
    c for c in required_unit_cols
    if c not in unit_index.columns
]
if missing:
    raise RuntimeError(
        f"Missing terrain unit-index columns: {missing}"
    )

approach_cols = [
    "unit_uid",
    "approach_azimuth_deg",
    "approach_direction_reliable",
    "approach_vs_line_angle_deg",
]

missing = [
    c for c in approach_cols
    if c not in approach.columns
]
if missing:
    raise RuntimeError(
        f"Missing 04A columns: {missing}"
    )

base = unit_index.copy()

base["Year"] = pd.to_numeric(
    base["Year"],
    errors="raise",
).astype(int)

base["landfire_version_key"] = (
    base["Year"].map(VERSION_BY_FIRE_YEAR)
)

# terrain_operational_unit_index_v1 may already carry one or more
# 04A direction fields. Drop those copies before merging the canonical
# 04A table; otherwise pandas creates *_x / *_y suffixed columns and the
# unsuffixed field names disappear.
overlap_04a_cols = [
    c
    for c in approach_cols
    if c != "unit_uid" and c in base.columns
]

print("Overlapping 04A columns already in terrain unit index:")
print(overlap_04a_cols)

base = base.drop(
    columns=overlap_04a_cols,
    errors="ignore",
)

base = base.merge(
    approach[approach_cols],
    on="unit_uid",
    how="left",
    validate="one_to_one",
)

# Canonical post-merge schema: no pandas suffix artifacts allowed.
suffix_artifacts = [
    c
    for c in base.columns
    if c.endswith("_x") or c.endswith("_y")
]

if suffix_artifacts:
    raise RuntimeError(
        f"Unexpected merge suffix columns remain: {suffix_artifacts}"
    )

for c in approach_cols:
    if c not in base.columns:
        raise RuntimeError(
            f"Canonical 04A column missing after merge: {c}"
        )

print("04A CANONICAL MERGE: PASS")

base["landfire_directional_context_valid"] = (
    base["approach_direction_reliable"]
    .fillna(False)
    & pd.to_numeric(
        base["approach_azimuth_deg"],
        errors="coerce",
    ).notna()
)

resolved_base = base[
    base["landfire_version_key"].isin(
        RESOLVED_VERSIONS
    )
].copy()

print("Operational units:", f"{len(base):,}")
print("Resolved-vintage units:", f"{len(resolved_base):,}")
print(
    "Frozen V5 fields:",
    f"{len(feature_dictionary):,}",
)

display(
    feature_dictionary.groupby(
        ["tier", "product"],
        as_index=False,
    ).agg(
        features=("feature", "size")
    )
)

## 3. Build full resolved-vintage block manifest

In [ ]:

production_base = resolved_base[
    resolved_base["landfire_version_key"].isin(
        PRODUCTION_VERSIONS
    )
].copy()

block_manifest = (
    production_base.groupby(
        [
            "landfire_version_key",
            "terrain_block_id",
            "terrain_block_ix",
            "terrain_block_iy",
        ],
        as_index=False,
    )
    .agg(
        units=("unit_uid", "size"),
        incidents=("_irwin_norm", "nunique"),
        directional_valid_units=(
            "landfire_directional_context_valid",
            "sum",
        ),
    )
    .sort_values(
        [
            "landfire_version_key",
            "terrain_block_id",
        ]
    )
    .reset_index(drop=True)
)

if MAX_BLOCKS_TOTAL is not None:
    block_manifest = block_manifest.head(
        int(MAX_BLOCKS_TOTAL)
    ).copy()

block_manifest["expected_product_tiles"] = len(PRODUCTS)

display(
    block_manifest.groupby(
        "landfire_version_key",
        as_index=False,
    ).agg(
        blocks=("terrain_block_id", "size"),
        units=("units", "sum"),
        incidents=("incidents", "sum"),
        expected_product_tiles=(
            "expected_product_tiles",
            "sum",
        ),
    )
)

print(
    "Total production blocks:",
    f"{len(block_manifest):,}",
)
print(
    "Total production units:",
    f"{int(block_manifest['units'].sum()):,}",
)
print(
    "Expected product tiles:",
    f"{int(block_manifest['expected_product_tiles'].sum()):,}",
)

block_manifest.to_csv(
    OUTPUT_ROOT / "PRODUCTION_BLOCK_MANIFEST_V3.csv",
    index=False,
)


## 4. V5 categorical and canopy contracts

In [ ]:
FBFM40_GROUP_CODES = {
    "NB": {91, 92, 93, 98, 99},
    "GR": set(range(101, 110)),
    "GS": set(range(121, 125)),
    "SH": set(range(141, 150)),
    "TU": set(range(161, 166)),
    "TL": set(range(181, 190)),
    "SB": set(range(201, 205)),
}

FBFM40_VALID_CODES = set().union(
    *FBFM40_GROUP_CODES.values()
)

FBFM40_NONBURNABLE_CODES = set(
    FBFM40_GROUP_CODES["NB"]
)

FBFM40_BURNABLE_CODES = (
    FBFM40_VALID_CODES
    - FBFM40_NONBURNABLE_CODES
)

assert len(FBFM40_VALID_CODES) == 45
assert len(FBFM40_BURNABLE_CODES) == 40
assert len(FBFM40_NONBURNABLE_CODES) == 5

FBFM40_CODE_TO_GROUP = {
    code: group
    for group, codes in FBFM40_GROUP_CODES.items()
    for code in codes
}

FBFM40_NB_SUBCODES = {
    "NB1": 91,
    "NB2": 92,
    "NB3": 93,
    "NB8": 98,
    "NB9": 99,
}

CANOPY_SPEC = {
    "CC": {
        "scale": 1.0,
        "top_censored_raw": None,
    },
    "CH": {
        "scale": 0.1,
        "top_censored_raw": 510,
    },
    "CBH": {
        "scale": 0.1,
        "top_censored_raw": 100,
    },
    "CBD": {
        "scale": 0.01,
        "top_censored_raw": 45,
    },
}

print("V5 PRODUCT SEMANTICS: PASS")

## 5. Weighted aggregation helpers

In [ ]:
def _clean_weighted(values, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)

    ok = (
        np.isfinite(v)
        & np.isfinite(w)
        & (w > 0)
    )

    return v[ok], w[ok]


def weighted_mean(values, weights):
    v, w = _clean_weighted(values, weights)

    if not len(v):
        return np.nan

    return float(
        np.average(v, weights=w)
    )


def weighted_std(values, weights):
    v, w = _clean_weighted(values, weights)

    if not len(v):
        return np.nan

    mu = np.average(v, weights=w)

    return float(
        np.sqrt(
            np.average(
                (v - mu) ** 2,
                weights=w,
            )
        )
    )


def weighted_quantile(values, weights, q):
    v, w = _clean_weighted(values, weights)

    if not len(v):
        return np.nan

    order = np.argsort(v)
    v = v[order]
    w = w[order]

    cw = np.cumsum(w)
    target = float(q) * cw[-1]

    idx = np.searchsorted(
        cw,
        target,
        side="left",
    )

    return float(v[idx])


def weighted_fraction(mask, weights):
    mask = np.asarray(mask, dtype=bool)
    w = np.asarray(weights, dtype=float)

    ok = np.isfinite(w) & (w > 0)

    if not ok.any():
        return np.nan

    return float(
        w[ok & mask].sum()
        / w[ok].sum()
    )


def weighted_entropy(categories, weights):
    c = np.asarray(categories)
    w = np.asarray(weights, dtype=float)

    ok = (
        pd.notna(c)
        & np.isfinite(w)
        & (w > 0)
    )

    if not ok.any():
        return np.nan

    c = c[ok]
    w = w[ok]

    totals = {}
    for value, weight in zip(c, w):
        totals[value] = (
            totals.get(value, 0.0)
            + float(weight)
        )

    p = np.asarray(
        list(totals.values()),
        dtype=float,
    )
    p = p / p.sum()

    return float(
        -(p * np.log(p)).sum()
    )


def jensen_shannon_divergence(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    if (
        len(p) != len(q)
        or len(p) == 0
        or not np.isfinite(p).all()
        or not np.isfinite(q).all()
        or p.sum() <= 0
        or q.sum() <= 0
    ):
        return np.nan

    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > 0
        return float(
            np.sum(
                a[mask]
                * np.log(
                    a[mask] / b[mask]
                )
            )
        )

    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

## 6. Build line-relative context requests

In [ ]:
def find_block_sample_path(block_id):
    direct = (
        TERRAIN_BLOCK_SAMPLE_ROOT /
        f"{block_id}.parquet"
    )

    if direct.exists():
        return direct

    candidates = list(
        TERRAIN_BLOCK_SAMPLE_ROOT.glob(
            f"*{block_id}*.parquet"
        )
    )

    if len(candidates) == 1:
        return candidates[0]

    raise RuntimeError(
        f"Could not resolve block sample shard for {block_id}. "
        f"Candidates={candidates}"
    )


def build_context_requests(
    line_samples,
    unit_direction,
):
    required = {
        "unit_uid",
        "x_m",
        "y_m",
        "sample_weight_m",
    }

    missing = sorted(
        required - set(line_samples.columns)
    )

    if missing:
        raise RuntimeError(
            f"Missing line-sample columns: {missing}"
        )

    x = line_samples.merge(
        unit_direction[
            [
                "unit_uid",
                "approach_azimuth_deg",
                "landfire_directional_context_valid",
            ]
        ],
        on="unit_uid",
        how="left",
        validate="many_to_one",
    ).copy()

    x["line_position_index"] = (
        x.groupby(
            "unit_uid",
            sort=False,
        ).cumcount()
    )

    theta = np.deg2rad(
        pd.to_numeric(
            x["approach_azimuth_deg"],
            errors="coerce",
        ).to_numpy(float)
    )

    east = np.sin(theta)
    north = np.cos(theta)

    frames = []

    line = x[
        [
            "unit_uid",
            "x_m",
            "y_m",
            "sample_weight_m",
            "line_position_index",
        ]
    ].copy()

    line["region"] = "line"
    line["offset_m"] = 0.0
    line["context_weight_m"] = (
        line["sample_weight_m"]
    )

    frames.append(line)

    valid = (
        x["landfire_directional_context_valid"]
        .fillna(False)
        .to_numpy(bool)
    )

    for region in sorted(DIRECTIONAL_REGIONS):
        depths = REGION_DEPTHS_M[region]

        for offset in depths:
            idx = np.flatnonzero(valid)

            g = x.loc[
                valid,
                [
                    "unit_uid",
                    "x_m",
                    "y_m",
                    "sample_weight_m",
                    "line_position_index",
                ],
            ].copy()

            g["x_m"] = (
                g["x_m"].to_numpy(float)
                + float(offset) * east[idx]
            )

            g["y_m"] = (
                g["y_m"].to_numpy(float)
                + float(offset) * north[idx]
            )

            g["region"] = region
            g["offset_m"] = float(offset)

            g["context_weight_m"] = (
                g["sample_weight_m"]
                / len(depths)
            )

            frames.append(g)

    out = pd.concat(
        frames,
        ignore_index=True,
    )

    out["request_id"] = np.arange(
        len(out),
        dtype=np.int64,
    )

    return out

## 7. ImageServer metadata and native-grid tile export

### Request-aware production tile envelope

The fixed block envelope is retained as provenance/documentation, but actual
production downloads use the full context-request extent plus two pixels of
padding.


In [ ]:
def request_bbox_from_context(
    requests_df,
    pad_m=REQUEST_BBOX_PAD_M,
):
    x = pd.to_numeric(
        requests_df["x_m"],
        errors="coerce",
    ).to_numpy(float)

    y = pd.to_numeric(
        requests_df["y_m"],
        errors="coerce",
    ).to_numpy(float)

    finite = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    if not finite.any():
        raise RuntimeError(
            "No finite LANDFIRE context coordinates in block."
        )

    xmin = float(np.min(x[finite]) - pad_m)
    ymin = float(np.min(y[finite]) - pad_m)
    xmax = float(np.max(x[finite]) + pad_m)
    ymax = float(np.max(y[finite]) + pad_m)

    if not (
        xmin < xmax
        and ymin < ymax
    ):
        raise RuntimeError(
            f"Invalid request bbox: {(xmin, ymin, xmax, ymax)}"
        )

    return (
        xmin,
        ymin,
        xmax,
        ymax,
    )


def cached_tile_covers_snapped_bbox(
    path,
    snapped_bbox,
    tolerance_m=1e-6,
):
    """
    Return True only when an existing cached raster fully covers the
    newly required snapped bbox on the expected EPSG:5070 / 30 m grid.
    """
    if not path.exists() or path.stat().st_size <= 0:
        return False

    xmin, ymin, xmax, ymax, _, _ = snapped_bbox

    try:
        with rasterio.open(path) as src:
            if (
                CRS.from_user_input(src.crs)
                != CRS.from_epsg(5070)
            ):
                return False

            if not (
                abs(abs(src.transform.a) - 30.0) <= 1e-9
                and abs(abs(src.transform.e) - 30.0) <= 1e-9
            ):
                return False

            b = src.bounds

            return bool(
                b.left <= xmin + tolerance_m
                and b.bottom <= ymin + tolerance_m
                and b.right >= xmax - tolerance_m
                and b.top >= ymax - tolerance_m
            )

    except Exception:
        return False


# Synthetic coverage QA.
from rasterio.transform import from_origin
from rasterio.io import MemoryFile

_test_arr = np.zeros(
    (10, 10),
    dtype=np.int16,
)

with MemoryFile() as _mem:
    with _mem.open(
        driver="GTiff",
        height=10,
        width=10,
        count=1,
        dtype="int16",
        crs="EPSG:5070",
        transform=from_origin(
            0.0,
            300.0,
            30.0,
            30.0,
        ),
        nodata=-9999,
    ) as _dst:
        _dst.write(
            _test_arr,
            1,
        )

    # MemoryFile does not expose a normal Path for the production helper,
    # so bbox math itself is checked separately here.
    _bbox = request_bbox_from_context(
        pd.DataFrame({
            "x_m": [100.0, 200.0],
            "y_m": [100.0, 200.0],
        }),
        pad_m=60.0,
    )

    assert _bbox == (
        40.0,
        40.0,
        260.0,
        260.0,
    )

print("REQUEST-AWARE TILE BBOX CONTRACT: PASS")


In [ ]:
def get_service_metadata(service_url):
    r = session.get(
        service_url,
        params={"f": "pjson"},
        timeout=REQUEST_TIMEOUT_S,
    )
    r.raise_for_status()

    payload = r.json()

    if "error" in payload:
        raise RuntimeError(payload["error"])

    return payload


def snap_bbox_to_service_grid(
    requested_bbox,
    service_meta,
):
    xmin, ymin, xmax, ymax = map(
        float,
        requested_bbox,
    )

    extent = service_meta["extent"]

    sxmin = float(extent["xmin"])
    symin = float(extent["ymin"])
    sxmax = float(extent["xmax"])
    symax = float(extent["ymax"])

    px = abs(float(service_meta["pixelSizeX"]))
    py = abs(float(service_meta["pixelSizeY"]))

    if not (
        abs(px - 30.0) <= 1e-9
        and abs(py - 30.0) <= 1e-9
    ):
        raise RuntimeError(
            f"Unexpected native pixel size: {px}, {py}"
        )

    axmin = max(
        sxmin,
        sxmin
        + math.floor(
            (xmin - sxmin) / px
        ) * px,
    )

    aymin = max(
        symin,
        symin
        + math.floor(
            (ymin - symin) / py
        ) * py,
    )

    axmax = min(
        sxmax,
        sxmin
        + math.ceil(
            (xmax - sxmin) / px
        ) * px,
    )

    aymax = min(
        symax,
        symin
        + math.ceil(
            (ymax - symin) / py
        ) * py,
    )

    width = int(
        round(
            (axmax - axmin) / px
        )
    )

    height = int(
        round(
            (aymax - aymin) / py
        )
    )

    if width <= 0 or height <= 0:
        raise RuntimeError(
            "Snapped tile has non-positive dimensions."
        )

    return (
        axmin,
        aymin,
        axmax,
        aymax,
        width,
        height,
    )


def safe_token(value, n=16):
    return hashlib.sha1(
        str(value).encode("utf-8")
    ).hexdigest()[:n]


def export_raw_tile(
    version_key,
    product,
    block_id,
    requested_bbox,
):
    service_url = (
        SERVICE_URLS[version_key][product]
    )

    meta = get_service_metadata(
        service_url
    )

    if meta.get("pixelType") != "S16":
        raise RuntimeError(
            f"{version_key}/{product}: "
            f"pixelType={meta.get('pixelType')}"
        )

    if int(meta.get("bandCount", -1)) != 1:
        raise RuntimeError(
            f"{version_key}/{product}: "
            f"bandCount={meta.get('bandCount')}"
        )

    snapped = snap_bbox_to_service_grid(
        requested_bbox,
        meta,
    )

    xmin, ymin, xmax, ymax, width, height = snapped

    out_dir = (
        CACHE_ROOT /
        version_key /
        product
    )
    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    out_path = (
        out_dir /
        f"{safe_token(block_id)}.tif"
    )

    if USE_TILE_CACHE and out_path.exists():
        if cached_tile_covers_snapped_bbox(
            out_path,
            snapped,
        ):
            return out_path, meta, snapped, True

        print(
            "    CACHE REFRESH:",
            version_key,
            product,
            block_id,
            "existing tile does not cover required request extent",
        )

    params = {
        "f": "json",
        "bbox": f"{xmin},{ymin},{xmax},{ymax}",
        "bboxSR": "5070",
        "imageSR": "5070",
        "size": f"{width},{height}",
        "format": "tiff",
        "pixelType": "S16",
        "interpolation": "RSP_NearestNeighbor",
        "compression": "LZ77",
        "noData": "-9999",
        "noDataInterpretation": "esriNoDataMatchAny",
        "adjustAspectRatio": "false",
        "mosaicRule": json.dumps(
            MOSAIC_RULE,
            separators=(",", ":"),
        ),
    }

    last_error = None

    # A fresh export href may be needed if ArcGIS expires/stages a temporary
    # output slowly. Do this inside the product, not by restarting the block.
    for export_issue in range(
        1,
        EXPORT_REISSUE_ATTEMPTS + 1,
    ):
        r = session.get(
            service_url + "/exportImage",
            params=params,
            timeout=REQUEST_TIMEOUT_S,
        )
        r.raise_for_status()

        payload = r.json()

        if "error" in payload:
            raise RuntimeError(
                payload["error"]
            )

        href = payload.get("href")

        if not href:
            raise RuntimeError(
                f"No exportImage href for {version_key}/{product}"
            )

        for href_attempt in range(
            1,
            HREF_DOWNLOAD_ATTEMPTS + 1,
        ):
            try:
                if href_attempt > 1:
                    time.sleep(
                        HREF_RETRY_SLEEP_S
                        * (href_attempt - 1)
                    )

                tile = session.get(
                    href,
                    timeout=REQUEST_TIMEOUT_S,
                )
                tile.raise_for_status()

                tmp_path = out_path.with_suffix(
                    out_path.suffix + ".part"
                )

                tmp_path.write_bytes(
                    tile.content
                )

                # Atomic replacement avoids treating a partially written file
                # as a valid cache item after interruption.
                tmp_path.replace(
                    out_path
                )

                return out_path, meta, snapped, False

            except requests.RequestException as e:
                last_error = e

        # The href itself may be stale/not-yet-staged. Ask exportImage for
        # another temporary href before escalating.
        if export_issue < EXPORT_REISSUE_ATTEMPTS:
            time.sleep(
                HREF_RETRY_SLEEP_S
                * export_issue
            )

    raise RuntimeError(
        f"{version_key}/{product}/{block_id}: "
        f"temporary export TIFF could not be downloaded after "
        f"{EXPORT_REISSUE_ATTEMPTS} export issue(s) × "
        f"{HREF_DOWNLOAD_ATTEMPTS} href attempt(s). "
        f"Last error: {last_error}"
    )


## 8. Tile validation and local sampling

In [ ]:
def validate_tile(
    path,
    product,
):
    with rasterio.open(path) as src:
        info = {
            "width": src.width,
            "height": src.height,
            "count": src.count,
            "dtype": src.dtypes[0],
            "crs": str(src.crs),
            "nodata": src.nodata,
            "pixel_x": abs(src.transform.a),
            "pixel_y": abs(src.transform.e),
            "left": src.bounds.left,
            "bottom": src.bounds.bottom,
            "right": src.bounds.right,
            "top": src.bounds.top,
            "transform_a": src.transform.a,
            "transform_b": src.transform.b,
            "transform_c": src.transform.c,
            "transform_d": src.transform.d,
            "transform_e": src.transform.e,
            "transform_f": src.transform.f,
        }

        raw = src.read(1)

    checks = {
        "count_ok": info["count"] == 1,
        "dtype_ok": str(info["dtype"]).lower() == "int16",
        "crs_ok": CRS.from_user_input(info["crs"]) == CRS.from_epsg(5070),
        "pixel_ok": (
            abs(info["pixel_x"] - 30.0) <= 1e-9
            and abs(info["pixel_y"] - 30.0) <= 1e-9
        ),
        "nodata_ok": (
            info["nodata"] is None
            or float(info["nodata"]) == -9999.0
        ),
    }

    valid = raw[
        raw != -9999
    ]

    if product == "FBFM40":
        domain_ok = bool(
            np.isin(
                valid,
                list(FBFM40_VALID_CODES),
            ).all()
        )
    elif product == "CC":
        domain_ok = bool(
            ((valid >= 0) & (valid <= 100)).all()
        )
    elif product == "CH":
        domain_ok = bool(
            ((valid >= 0) & (valid <= 510)).all()
        )
    elif product == "CBH":
        domain_ok = bool(
            ((valid >= 0) & (valid <= 100)).all()
        )
    elif product == "CBD":
        domain_ok = bool(
            ((valid >= 0) & (valid <= 45)).all()
        )
    else:
        raise KeyError(product)

    checks["domain_ok"] = domain_ok

    if not all(checks.values()):
        raise RuntimeError(
            f"Tile QA failed for {path}: {checks}"
        )

    return info, checks


def sample_tile_at_requests(
    tile_path,
    requests_df,
    product,
):
    with rasterio.open(tile_path) as src:
        arr = src.read(1)

        xs = requests_df["x_m"].to_numpy(float)
        ys = requests_df["y_m"].to_numpy(float)

        rows, cols = rasterio.transform.rowcol(
            src.transform,
            xs,
            ys,
        )

        rows = np.asarray(rows, dtype=np.int64)
        cols = np.asarray(cols, dtype=np.int64)

        inside = (
            (rows >= 0)
            & (rows < src.height)
            & (cols >= 0)
            & (cols < src.width)
        )

        if not inside.all():
            bad = int((~inside).sum())
            raise RuntimeError(
                f"{product}: {bad} requests fell outside cached tile."
            )

        values = arr[
            rows,
            cols,
        ].astype(np.int32)

        # Physical center of the selected native raster cell.
        center_x, center_y = rasterio.transform.xy(
            src.transform,
            rows,
            cols,
            offset="center",
        )

        center_x = np.asarray(
            center_x,
            dtype=float,
        )

        center_y = np.asarray(
            center_y,
            dtype=float,
        )

    return pd.DataFrame({
        "request_id": requests_df["request_id"].to_numpy(np.int64),
        f"{product}_row": rows,
        f"{product}_col": cols,
        f"{product}_cell_center_x_m": center_x,
        f"{product}_cell_center_y_m": center_y,
        f"{product}_raw": values,
    })

## 9. Collapse repeated native cells

In [ ]:
def collapse_product_cells(
    sampled_requests,
    product,
):
    row_col = [
        f"{product}_row",
        f"{product}_col",
    ]

    required = [
        "unit_uid",
        "region",
        "context_weight_m",
        f"{product}_raw",
        *row_col,
    ]

    missing = [
        c for c in required
        if c not in sampled_requests.columns
    ]

    if missing:
        raise RuntimeError(
            f"Missing columns for {product}: {missing}"
        )

    value_check = (
        sampled_requests.groupby(
            [
                "unit_uid",
                "region",
                *row_col,
            ],
            sort=False,
        )[f"{product}_raw"]
        .nunique(dropna=False)
    )

    if (value_check > 1).any():
        raise RuntimeError(
            f"{product}: one native cell produced multiple raw values."
        )

    out = (
        sampled_requests.groupby(
            [
                "unit_uid",
                "region",
                *row_col,
            ],
            as_index=False,
            sort=False,
        )
        .agg(
            raw_value=(f"{product}_raw", "first"),
            context_weight_m=("context_weight_m", "sum"),
            contributing_requests=("request_id", "size"),
        )
    )

    return out

## 10. Region-level FBFM40 and canopy aggregation

In [ ]:
def aggregate_fbfm40_group(g):
    raw = pd.to_numeric(
        g["raw_value"],
        errors="coerce",
    ).to_numpy(float)

    w = g[
        "context_weight_m"
    ].to_numpy(float)

    valid = (
        np.isfinite(raw)
        & (raw != -9999)
        & np.isin(
            raw.astype(
                int,
                copy=False,
            ),
            list(FBFM40_VALID_CODES),
        )
    )

    out = {
        "valid_weight_fraction": weighted_fraction(
            valid,
            w,
        ),
        "unique_cell_count": int(
            len(g)
        ),
    }

    if not valid.any():
        for group in FBFM40_GROUP_CODES:
            out[f"group_{group}_fraction"] = np.nan

        for nb_name in FBFM40_NB_SUBCODES:
            out[f"{nb_name.lower()}_fraction"] = np.nan

        out["group_entropy"] = np.nan
        out["exact_entropy"] = np.nan
        out["unique_exact_models"] = np.nan
        out["dominant_group_fraction"] = np.nan

        for code_value in sorted(
            FBFM40_VALID_CODES
        ):
            out[
                f"exact_code_{code_value}_fraction"
            ] = np.nan

        return out

    vr = raw[valid].astype(int)
    vw = w[valid]

    groups = np.asarray([
        FBFM40_CODE_TO_GROUP[int(code)]
        for code in vr
    ])

    group_fracs = []

    for group in FBFM40_GROUP_CODES:
        frac = weighted_fraction(
            groups == group,
            vw,
        )

        out[
            f"group_{group}_fraction"
        ] = frac

        group_fracs.append(frac)

    for nb_name, nb_code in (
        FBFM40_NB_SUBCODES.items()
    ):
        out[
            f"{nb_name.lower()}_fraction"
        ] = weighted_fraction(
            vr == nb_code,
            vw,
        )

    out["group_entropy"] = weighted_entropy(
        groups,
        vw,
    )

    out["exact_entropy"] = weighted_entropy(
        vr,
        vw,
    )

    out["unique_exact_models"] = int(
        len(np.unique(vr))
    )

    out["dominant_group_fraction"] = float(
        np.nanmax(group_fracs)
    )

    for code_value in sorted(
        FBFM40_VALID_CODES
    ):
        out[
            f"exact_code_{code_value}_fraction"
        ] = weighted_fraction(
            vr == code_value,
            vw,
        )

    return out


def aggregate_canopy_group(
    g,
    product,
):
    spec = CANOPY_SPEC[product]

    raw = pd.to_numeric(
        g["raw_value"],
        errors="coerce",
    ).to_numpy(
        dtype=float,
        copy=True,
    )

    w = g[
        "context_weight_m"
    ].to_numpy(float)

    valid = (
        np.isfinite(raw)
        & (raw != -9999)
    )

    out = {
        "valid_weight_fraction": weighted_fraction(
            valid,
            w,
        ),
        "unique_cell_count": int(
            len(g)
        ),
    }

    stats = [
        "mean",
        "std",
        "p10",
        "median",
        "p90",
        "zero_fraction",
        "positive_mean",
        "positive_std",
        "positive_p10",
        "positive_median",
        "positive_p90",
        "top_censored_fraction",
    ]

    if not valid.any():
        for stat in stats:
            out[stat] = np.nan
        return out

    vr = raw[valid]
    vw = w[valid]

    physical = (
        vr
        * float(spec["scale"])
    )

    out["mean"] = weighted_mean(
        physical,
        vw,
    )

    out["std"] = weighted_std(
        physical,
        vw,
    )

    out["p10"] = weighted_quantile(
        physical,
        vw,
        0.10,
    )

    out["median"] = weighted_quantile(
        physical,
        vw,
        0.50,
    )

    out["p90"] = weighted_quantile(
        physical,
        vw,
        0.90,
    )

    out["zero_fraction"] = weighted_fraction(
        vr == 0,
        vw,
    )

    positive = vr > 0

    if positive.any():
        vp = physical[positive]
        wp = vw[positive]

        out["positive_mean"] = weighted_mean(vp, wp)
        out["positive_std"] = weighted_std(vp, wp)
        out["positive_p10"] = weighted_quantile(vp, wp, 0.10)
        out["positive_median"] = weighted_quantile(vp, wp, 0.50)
        out["positive_p90"] = weighted_quantile(vp, wp, 0.90)
    else:
        out["positive_mean"] = np.nan
        out["positive_std"] = np.nan
        out["positive_p10"] = np.nan
        out["positive_median"] = np.nan
        out["positive_p90"] = np.nan

    top = spec[
        "top_censored_raw"
    ]

    out["top_censored_fraction"] = (
        weighted_fraction(
            vr == top,
            vw,
        )
        if top is not None
        else np.nan
    )

    return out

## 11. Joint canopy-derived features

In [ ]:
def build_joint_canopy_cells(
    sampled_requests,
    center_tolerance_m=1e-6,
):
    """
    Build joint CH/CBH/CBD native-cell table.

    Critical contract:
      the three products must select the same PHYSICAL native-cell center
      for every request. Local tile row/col equality is not required.
    """

    required = []

    for product in ["CH", "CBH", "CBD"]:
        required.extend([
            f"{product}_cell_center_x_m",
            f"{product}_cell_center_y_m",
            f"{product}_raw",
        ])

    missing = [
        c for c in required
        if c not in sampled_requests.columns
    ]

    if missing:
        raise RuntimeError(
            f"Missing joint-canopy columns: {missing}"
        )

    ch_x = sampled_requests[
        "CH_cell_center_x_m"
    ].to_numpy(float)

    ch_y = sampled_requests[
        "CH_cell_center_y_m"
    ].to_numpy(float)

    alignment_rows = []

    for product in ["CBH", "CBD"]:
        px = sampled_requests[
            f"{product}_cell_center_x_m"
        ].to_numpy(float)

        py = sampled_requests[
            f"{product}_cell_center_y_m"
        ].to_numpy(float)

        dx = np.abs(px - ch_x)
        dy = np.abs(py - ch_y)

        max_dx = float(
            np.nanmax(dx)
        )

        max_dy = float(
            np.nanmax(dy)
        )

        alignment_rows.append({
            "reference_product": "CH",
            "comparison_product": product,
            "max_abs_center_dx_m": max_dx,
            "max_abs_center_dy_m": max_dy,
            "all_centers_aligned": bool(
                (dx <= center_tolerance_m).all()
                and (dy <= center_tolerance_m).all()
            ),
        })

    alignment = pd.DataFrame(
        alignment_rows
    )

    if not alignment[
        "all_centers_aligned"
    ].all():
        raise RuntimeError(
            "CH/CBH/CBD native cell centers are not aligned.\\n"
            + alignment.to_string(index=False)
        )

    x = sampled_requests[
        [
            "unit_uid",
            "region",
            "context_weight_m",
            "CH_cell_center_x_m",
            "CH_cell_center_y_m",
            "CH_raw",
            "CBH_raw",
            "CBD_raw",
        ]
    ].copy()

    # Use CH physical center as canonical joint-canopy cell identity because
    # alignment to CBH/CBD was asserted above.
    x = x.rename(
        columns={
            "CH_cell_center_x_m": "cell_center_x_m",
            "CH_cell_center_y_m": "cell_center_y_m",
        }
    )

    value_cols = [
        "CH_raw",
        "CBH_raw",
        "CBD_raw",
    ]

    for value_col in value_cols:
        n = (
            x.groupby(
                [
                    "unit_uid",
                    "region",
                    "cell_center_x_m",
                    "cell_center_y_m",
                ],
                sort=False,
            )[value_col]
            .nunique(dropna=False)
        )

        if (n > 1).any():
            raise RuntimeError(
                f"Multiple {value_col} values at one physical native cell."
            )

    cells = (
        x.groupby(
            [
                "unit_uid",
                "region",
                "cell_center_x_m",
                "cell_center_y_m",
            ],
            as_index=False,
            sort=False,
        )
        .agg(
            context_weight_m=(
                "context_weight_m",
                "sum",
            ),
            CH_raw=("CH_raw", "first"),
            CBH_raw=("CBH_raw", "first"),
            CBD_raw=("CBD_raw", "first"),
        )
    )

    ch = cells["CH_raw"].to_numpy(float)
    cbh = cells["CBH_raw"].to_numpy(float)
    cbd = cells["CBD_raw"].to_numpy(float)

    joint_valid = (
        np.isfinite(ch)
        & np.isfinite(cbh)
        & np.isfinite(cbd)
        & (ch != -9999)
        & (cbh != -9999)
        & (cbd != -9999)
    )

    ch_m = np.full(
        len(cells),
        np.nan,
    )

    cbh_m = np.full(
        len(cells),
        np.nan,
    )

    cbd_phys = np.full(
        len(cells),
        np.nan,
    )

    ch_m[joint_valid] = (
        ch[joint_valid] / 10.0
    )

    cbh_m[joint_valid] = (
        cbh[joint_valid] / 10.0
    )

    cbd_phys[joint_valid] = (
        cbd[joint_valid] / 100.0
    )

    inconsistent = (
        joint_valid
        & (cbh_m > ch_m)
    )

    depth = np.full(
        len(cells),
        np.nan,
    )

    depth[joint_valid] = np.maximum(
        ch_m[joint_valid]
        - cbh_m[joint_valid],
        0.0,
    )

    cfl = np.full(
        len(cells),
        np.nan,
    )

    cfl[joint_valid] = (
        cbd_phys[joint_valid]
        * depth[joint_valid]
    )

    cells["joint_valid"] = joint_valid
    cells["cbh_gt_ch"] = inconsistent
    cells["crown_depth_m"] = depth
    cells["cfl_kg_m2"] = cfl

    return cells, alignment

def aggregate_joint_canopy_group(g):
    w = g[
        "context_weight_m"
    ].to_numpy(float)

    valid = g[
        "joint_valid"
    ].to_numpy(bool)

    out = {
        "joint_valid_weight_fraction": weighted_fraction(
            valid,
            w,
        ),
        "cbh_gt_ch_fraction": weighted_fraction(
            g["cbh_gt_ch"].to_numpy(bool),
            w,
        ),
    }

    if not valid.any():
        out.update({
            "crown_depth_mean": np.nan,
            "crown_depth_p90": np.nan,
            "cfl_mean": np.nan,
            "cfl_p90": np.nan,
        })
        return out

    vw = w[valid]

    depth = g.loc[
        valid,
        "crown_depth_m",
    ].to_numpy(float)

    cfl = g.loc[
        valid,
        "cfl_kg_m2",
    ].to_numpy(float)

    out["crown_depth_mean"] = weighted_mean(
        depth,
        vw,
    )
    out["crown_depth_p90"] = weighted_quantile(
        depth,
        vw,
        0.90,
    )
    out["cfl_mean"] = weighted_mean(
        cfl,
        vw,
    )
    out["cfl_p90"] = weighted_quantile(
        cfl,
        vw,
        0.90,
    )

    return out

## 12. Feature-name mapping helpers

In [ ]:

def _emit_if_frozen(out, name, value):
    """
    Emit only exact fields frozen by 04D1C V5.

    Aggregation helpers are allowed to compute richer intermediate statistics,
    but production output must match the frozen feature dictionary exactly.
    """
    if name in FROZEN_V5_FEATURE_SET:
        out[name] = value


def prefix_region_stats(
    unit_uid,
    region,
    product,
    stats,
):
    out = {"unit_uid": unit_uid}

    if product == "FBFM40":
        for key, value in stats.items():

            if key == "valid_weight_fraction":
                name = (
                    f"landfire_{region}_fbfm40_"
                    "valid_weight_fraction"
                )

            elif key == "unique_cell_count":
                name = (
                    f"landfire_{region}_fbfm40_"
                    "unique_cell_count"
                )

            # IMPORTANT: group_entropy is a statistic, not a group fraction.
            elif key == "group_entropy":
                name = (
                    f"landfire_{region}_"
                    "fbfm_group_entropy"
                )

            elif (
                key.startswith("group_")
                and key.endswith("_fraction")
            ):
                suffix = (
                    key[len("group_"):]
                    .removesuffix("_fraction")
                )

                name = (
                    f"landfire_{region}_fbfm_group_"
                    f"{suffix.lower()}_fraction"
                )

            elif key.startswith("nb"):
                name = (
                    f"landfire_{region}_fbfm_{key}"
                )

            elif key.startswith("exact_code_"):
                name = (
                    f"landfire_{region}_fbfm_"
                    f"{key.replace('exact_', '')}"
                )

            else:
                name = (
                    f"landfire_{region}_fbfm_{key}"
                )

            _emit_if_frozen(
                out,
                name,
                value,
            )

    elif product in CANOPY_SPEC:
        for key, value in stats.items():
            name = (
                f"landfire_{region}_"
                f"{product.lower()}_{key}"
            )

            _emit_if_frozen(
                out,
                name,
                value,
            )

    elif product == "DERIVED_CANOPY":
        for key, value in stats.items():

            if key == "joint_valid_weight_fraction":
                name = (
                    f"landfire_{region}_"
                    "derived_canopy_joint_valid_weight_fraction"
                )
            else:
                name = (
                    f"landfire_{region}_{key}"
                )

            _emit_if_frozen(
                out,
                name,
                value,
            )

    else:
        raise KeyError(product)

    return out


# ------------------------------------------------------------------
# Static naming QA for the exact V3 failure mode.
# ------------------------------------------------------------------
_test_stats = {
    "group_entropy": 0.5,
    "group_GR_fraction": 0.7,
    "valid_weight_fraction": 1.0,
    "unique_cell_count": 10,
}

_test_named = prefix_region_stats(
    "qa_unit",
    "line",
    "FBFM40",
    _test_stats,
)

assert (
    "landfire_line_fbfm_group_entropy"
    in _test_named
)

assert (
    "landfire_line_fbfm_group_entropy_fraction"
    not in _test_named
)

# Internal canopy statistics not frozen in V5 must not leak into output.
_canopy_test = prefix_region_stats(
    "qa_unit",
    "line",
    "CBD",
    {
        "mean": 0.1,
        "std": 0.2,       # internal, not V5 for CBD
        "median": 0.1,    # internal, not V5 for CBD
        "positive_mean": 0.2,
    },
)

assert "landfire_line_cbd_mean" in _canopy_test
assert "landfire_line_cbd_positive_mean" in _canopy_test
assert "landfire_line_cbd_std" not in _canopy_test
assert "landfire_line_cbd_median" not in _canopy_test

print("FROZEN-V5 FEATURE-NAME EMISSION CONTRACT: PASS")


## 13. Aggregate one block into region features

In [ ]:
def aggregate_region_features(
    sampled_requests,
):
    unit_rows = {}

    def update(row):
        uid = row.pop("unit_uid")
        unit_rows.setdefault(
            uid,
            {"unit_uid": uid},
        ).update(row)

    # FBFM40.
    collapsed = collapse_product_cells(
        sampled_requests,
        "FBFM40",
    )

    for (uid, region), g in collapsed.groupby(
        ["unit_uid", "region"],
        sort=False,
    ):
        stats = aggregate_fbfm40_group(g)

        update(
            prefix_region_stats(
                uid,
                region,
                "FBFM40",
                stats,
            )
        )

    # Canopy products.
    for product in [
        "CC",
        "CH",
        "CBH",
        "CBD",
    ]:
        collapsed = collapse_product_cells(
            sampled_requests,
            product,
        )

        for (uid, region), g in collapsed.groupby(
            ["unit_uid", "region"],
            sort=False,
        ):
            stats = aggregate_canopy_group(
                g,
                product,
            )

            update(
                prefix_region_stats(
                    uid,
                    region,
                    product,
                    stats,
                )
            )

    # Joint canopy: requires CH/CBH/CBD physical cell-center alignment.
    joint, canopy_alignment_qa = build_joint_canopy_cells(
        sampled_requests,
    )

    for (uid, region), g in joint.groupby(
        ["unit_uid", "region"],
        sort=False,
    ):
        stats = aggregate_joint_canopy_group(g)

        update(
            prefix_region_stats(
                uid,
                region,
                "DERIVED_CANOPY",
                stats,
            )
        )

    return (
        pd.DataFrame(
            unit_rows.values()
        ),
        canopy_alignment_qa,
    )

## 14. Fireward/protected contrast features

In [ ]:
FBFM_GROUP_ORDER = list(
    FBFM40_GROUP_CODES.keys()
)


def add_contrast_features(features):
    out = features.copy()

    for scale in ["near", "far"]:
        f = f"fireward_{scale}"
        p = f"protected_{scale}"

        # JSD broad-group composition.
        js_values = []

        for _, row in out.iterrows():
            fv = np.asarray([
                row.get(
                    f"landfire_{f}_fbfm_group_{g.lower()}_fraction",
                    np.nan,
                )
                for g in FBFM_GROUP_ORDER
            ], dtype=float)

            pv = np.asarray([
                row.get(
                    f"landfire_{p}_fbfm_group_{g.lower()}_fraction",
                    np.nan,
                )
                for g in FBFM_GROUP_ORDER
            ], dtype=float)

            js_values.append(
                jensen_shannon_divergence(
                    fv,
                    pv,
                )
            )

        out[
            f"landfire_{scale}_fbfm_group_js_divergence"
        ] = js_values

        pairs = [
            (
                "cc_positive_mean",
                f"landfire_{f}_cc_positive_mean",
                f"landfire_{p}_cc_positive_mean",
            ),
            (
                "cbd_positive_mean",
                f"landfire_{f}_cbd_positive_mean",
                f"landfire_{p}_cbd_positive_mean",
            ),
            (
                "crown_depth_mean",
                f"landfire_{f}_crown_depth_mean",
                f"landfire_{p}_crown_depth_mean",
            ),
            (
                "cfl_mean",
                f"landfire_{f}_cfl_mean",
                f"landfire_{p}_cfl_mean",
            ),
            (
                "fbfm_nb8_fraction",
                f"landfire_{f}_fbfm_nb8_fraction",
                f"landfire_{p}_fbfm_nb8_fraction",
            ),
            (
                "fbfm_nb9_fraction",
                f"landfire_{f}_fbfm_nb9_fraction",
                f"landfire_{p}_fbfm_nb9_fraction",
            ),
        ]

        for quantity, fc, pc in pairs:
            out[
                f"landfire_{scale}_fireward_minus_protected_{quantity}"
            ] = (
                pd.to_numeric(
                    out.get(fc),
                    errors="coerce",
                )
                - pd.to_numeric(
                    out.get(pc),
                    errors="coerce",
                )
            )

    return out

## 15. Positional p90/max fireward features

In [ ]:
def add_positional_features(
    features,
    sampled_requests,
):
    out = features.copy()

    # Build per-request physical CBD and derived CFL.
    x = sampled_requests.copy()

    cbd_raw = pd.to_numeric(
        x["CBD_raw"],
        errors="coerce",
    ).to_numpy(float)

    ch_raw = pd.to_numeric(
        x["CH_raw"],
        errors="coerce",
    ).to_numpy(float)

    cbh_raw = pd.to_numeric(
        x["CBH_raw"],
        errors="coerce",
    ).to_numpy(float)

    cbd_valid = (
        np.isfinite(cbd_raw)
        & (cbd_raw != -9999)
    )

    joint_valid = (
        cbd_valid
        & np.isfinite(ch_raw)
        & np.isfinite(cbh_raw)
        & (ch_raw != -9999)
        & (cbh_raw != -9999)
    )

    x["CBD_phys"] = np.nan
    x.loc[
        cbd_valid,
        "CBD_phys",
    ] = (
        cbd_raw[cbd_valid] / 100.0
    )

    x["CFL_phys"] = np.nan

    depth = np.maximum(
        ch_raw[joint_valid] / 10.0
        - cbh_raw[joint_valid] / 10.0,
        0.0,
    )

    x.loc[
        joint_valid,
        "CFL_phys",
    ] = (
        cbd_raw[joint_valid] / 100.0
        * depth
    )

    positional_rows = []

    for region in [
        "fireward_near",
        "fireward_far",
    ]:
        g = x[
            x["region"] == region
        ].copy()

        # One quantity per originating along-line position.
        per_pos_rows = []

        for (
            uid,
            pos,
        ), z in g.groupby(
            [
                "unit_uid",
                "line_position_index",
            ],
            sort=False,
        ):
            # The original line position carries one arc-length weight.
            pos_weight = float(
                z["sample_weight_m"].iloc[0]
            )

            cbd_values = z[
                "CBD_phys"
            ].to_numpy(float)

            cfl_values = z[
                "CFL_phys"
            ].to_numpy(float)

            cbd_pos = cbd_values[
                np.isfinite(cbd_values)
                & (cbd_values > 0)
            ]

            cfl_pos = cfl_values[
                np.isfinite(cfl_values)
            ]

            per_pos_rows.append({
                "unit_uid": uid,
                "line_position_index": pos,
                "position_weight_m": pos_weight,
                "cbd_positive_mean": (
                    float(np.mean(cbd_pos))
                    if len(cbd_pos)
                    else np.nan
                ),
                "cfl_mean": (
                    float(np.mean(cfl_pos))
                    if len(cfl_pos)
                    else np.nan
                ),
            })

        # A block may contain no reliable directional context at all.
        # In that case this region legitimately has no positional rows.
        if not per_pos_rows:
            continue

        per_pos = pd.DataFrame(
            per_pos_rows
        )

        scale = (
            "near"
            if region.endswith("near")
            else "far"
        )

        for uid, z in per_pos.groupby(
            "unit_uid",
            sort=False,
        ):
            row = {"unit_uid": uid}

            for quantity in [
                "cbd_positive_mean",
                "cfl_mean",
            ]:
                vals = z[
                    quantity
                ].to_numpy(float)

                weights = z[
                    "position_weight_m"
                ].to_numpy(float)

                good = (
                    np.isfinite(vals)
                    & np.isfinite(weights)
                    & (weights > 0)
                )

                if good.any():
                    p90 = weighted_quantile(
                        vals[good],
                        weights[good],
                        0.90,
                    )
                    vmax = float(
                        np.max(vals[good])
                    )
                else:
                    p90 = np.nan
                    vmax = np.nan

                row[
                    f"landfire_fireward_{scale}_position_p90_{quantity}"
                ] = p90

                row[
                    f"landfire_fireward_{scale}_position_max_{quantity}"
                ] = vmax

            positional_rows.append(row)

    positional = pd.DataFrame(
        positional_rows
    )

    if positional.empty:
        return out

    positional = (
        positional.groupby(
            "unit_uid",
            as_index=False,
        )
        .first()
    )

    return out.merge(
        positional,
        on="unit_uid",
        how="left",
        validate="one_to_one",
    )

# Synthetic QA for the exact V1 failure mode: no directional rows.
_positional_zero_direction_features = pd.DataFrame({
    "unit_uid": ["u0"],
    "landfire_line_cbd_mean": [0.0],
})

_positional_zero_direction_sampled = pd.DataFrame({
    "unit_uid": ["u0"],
    "region": ["line"],
    "line_position_index": [0],
    "sample_weight_m": [25.0],
    "CBD_raw": [0],
    "CH_raw": [0],
    "CBH_raw": [0],
})

_positional_zero_direction_out = add_positional_features(
    _positional_zero_direction_features,
    _positional_zero_direction_sampled,
)

assert len(_positional_zero_direction_out) == 1
assert "unit_uid" in _positional_zero_direction_out.columns

print("ZERO-DIRECTION POSITIONAL CONTRACT: PASS")


## 16. Run one production block

In [ ]:
def block_bbox_from_manifest_row(row):
    core_minx = (
        float(row["terrain_block_ix"])
        * BLOCK_SIZE_M
    )

    core_miny = (
        float(row["terrain_block_iy"])
        * BLOCK_SIZE_M
    )

    core_maxx = (
        core_minx
        + BLOCK_SIZE_M
    )

    core_maxy = (
        core_miny
        + BLOCK_SIZE_M
    )

    return (
        core_minx - BLOCK_HALO_M,
        core_miny - BLOCK_HALO_M,
        core_maxx + BLOCK_HALO_M,
        core_maxy + BLOCK_HALO_M,
    )


def run_production_block(row):
    version_key = row[
        "landfire_version_key"
    ]
    block_id = row[
        "terrain_block_id"
    ]

    print()
    print("=" * 100)
    print(
        "BLOCK:",
        version_key,
        block_id,
    )
    print("=" * 100)

    block_units = resolved_base[
        (resolved_base["landfire_version_key"] == version_key)
        & (resolved_base["terrain_block_id"] == block_id)
    ].copy()

    sample_path = find_block_sample_path(
        block_id
    )

    line_samples = pd.read_parquet(
        sample_path
    )

    line_samples = line_samples[
        line_samples["unit_uid"].isin(
            block_units["unit_uid"]
        )
    ].copy()

    if line_samples.empty:
        raise RuntimeError(
            f"No line samples for {block_id}"
        )

    requests_df = build_context_requests(
        line_samples,
        block_units[
            [
                "unit_uid",
                "approach_azimuth_deg",
                "landfire_directional_context_valid",
            ]
        ],
    )

    # Download envelope is derived from the ACTUAL frozen context
    # requests. This prevents edge units from exceeding a fixed block halo.
    requested_bbox = request_bbox_from_context(
        requests_df,
        pad_m=REQUEST_BBOX_PAD_M,
    )

    fixed_block_bbox = block_bbox_from_manifest_row(
        row
    )

    request_exceeds_fixed_block_bbox = not (
        requested_bbox[0] >= fixed_block_bbox[0]
        and requested_bbox[1] >= fixed_block_bbox[1]
        and requested_bbox[2] <= fixed_block_bbox[2]
        and requested_bbox[3] <= fixed_block_bbox[3]
    )

    if request_exceeds_fixed_block_bbox:
        print(
            "    NOTE: actual context extent exceeds fixed 360 m block halo; "
            "using adaptive request-aware tile bbox."
        )

    tile_rows = []
    sampled = requests_df.copy()

    for product in PRODUCTS:
        path, meta, snapped, cache_hit = (
            export_raw_tile(
                version_key,
                product,
                block_id,
                requested_bbox,
            )
        )

        info, checks = validate_tile(
            path,
            product,
        )

        s = sample_tile_at_requests(
            path,
            requests_df,
            product,
        )

        sampled = sampled.merge(
            s,
            on="request_id",
            how="left",
            validate="one_to_one",
        )

        tile_rows.append({
            "version_key": version_key,
            "block_id": block_id,
            "product": product,
            "cache_hit": cache_hit,
            "tile_path": str(path),
            "request_bbox_xmin": requested_bbox[0],
            "request_bbox_ymin": requested_bbox[1],
            "request_bbox_xmax": requested_bbox[2],
            "request_bbox_ymax": requested_bbox[3],
            "request_exceeds_fixed_360m_halo": (
                request_exceeds_fixed_block_bbox
            ),
            **info,
            **checks,
        })

    # Cross-product PHYSICAL cell-center diagnostics.
    # Local row/col indices are tile-relative and are not expected to match
    # when separately exported tiles have different local origins.
    center_diag_rows = []

    ref_x = sampled[
        "FBFM40_cell_center_x_m"
    ].to_numpy(float)

    ref_y = sampled[
        "FBFM40_cell_center_y_m"
    ].to_numpy(float)

    for product in PRODUCTS[1:]:
        px = sampled[
            f"{product}_cell_center_x_m"
        ].to_numpy(float)

        py = sampled[
            f"{product}_cell_center_y_m"
        ].to_numpy(float)

        dx = np.abs(px - ref_x)
        dy = np.abs(py - ref_y)

        center_diag_rows.append({
            "reference_product": "FBFM40",
            "comparison_product": product,
            "max_abs_center_dx_m": float(np.nanmax(dx)),
            "max_abs_center_dy_m": float(np.nanmax(dy)),
            "all_centers_aligned_1e6m": bool(
                (dx <= 1e-6).all()
                and (dy <= 1e-6).all()
            ),
        })

    product_center_qa = pd.DataFrame(
        center_diag_rows
    )

    features, canopy_alignment_qa = aggregate_region_features(
        sampled
    )

    features = add_contrast_features(
        features
    )

    features = add_positional_features(
        features,
        sampled,
    )

    # Unit metadata/provenance.
    meta_cols = [
        "unit_uid",
        "_irwin_norm",
        "IncidentName",
        "Year",
        "_engagement_norm",
        "terrain_block_id",
        "landfire_version_key",
        "landfire_directional_context_valid",
        "approach_vs_line_angle_deg",
    ]

    features = block_units[
        meta_cols
    ].merge(
        features,
        on="unit_uid",
        how="left",
        validate="one_to_one",
    )

    # Provenance status is QA-only.
    features[
        "landfire_source_status"
    ] = "resolved"

    # Ensure intentionally missing directional regions remain missing
    # for unreliable direction units.
    directional_cols = [
        c for c in features.columns
        if (
            "fireward_" in c
            or "protected_" in c
            or "_near_" in c
            or "_far_" in c
        )
        and c.startswith("landfire_")
    ]

    unreliable = ~features[
        "landfire_directional_context_valid"
    ].fillna(False)

    # Directional columns should already be NaN because no requests were
    # generated; assert rather than forcibly masking unexpected values.
    bad_nonmissing = (
        features.loc[
            unreliable,
            directional_cols,
        ]
        .notna()
        .any(axis=1)
    )

    if bad_nonmissing.any():
        raise RuntimeError(
            "Unreliable-direction units unexpectedly received directional features."
        )

    # V5 schema coverage.
    v5_features = set(
        feature_dictionary["feature"]
    )

    produced = set(
        features.columns
    )

    missing_v5 = sorted(
        v5_features - produced
    )

    unexpected_landfire = sorted(
        c for c in produced
        if c.startswith("landfire_")
        and c not in v5_features
    )

    shard = (
        SHARD_ROOT /
        f"{version_key}__{safe_token(block_id)}.parquet"
    )

    features.to_parquet(
        shard,
        index=False,
    )

    tile_qa = pd.DataFrame(
        tile_rows
    )

    tile_qa[
        "all_context_requests_inside_tile"
    ] = True

    return {
        "version_key": version_key,
        "block_id": block_id,
        "units": len(block_units),
        "line_samples": len(line_samples),
        "context_requests": len(requests_df),
        "feature_rows": len(features),
        "feature_columns": len(features.columns),
        "missing_v5_features": missing_v5,
        "unexpected_landfire_features": unexpected_landfire,
        "shard_path": str(shard),
        "features": features,
        "tile_qa": tile_qa,
        "product_center_qa": product_center_qa,
        "canopy_alignment_qa": canopy_alignment_qa,
    }


## 16B. Restart / shard-validation helpers

A block is considered complete only when its existing shard:

- exists and can be read;
- contains exactly the expected unit count;
- has unique `unit_uid`;
- contains every frozen V5 feature;
- contains no unexpected `landfire_*` feature;
- contains the expected vintage and block identity.

Invalid/incomplete shards are recomputed.


In [ ]:

def expected_shard_path(
    version_key,
    block_id,
):
    return (
        SHARD_ROOT /
        f"{version_key}__{safe_token(block_id)}.parquet"
    )


def validate_existing_shard(
    row,
):
    version_key = row["landfire_version_key"]
    block_id = row["terrain_block_id"]
    expected_units = int(row["units"])

    shard = expected_shard_path(
        version_key,
        block_id,
    )

    if not shard.exists():
        return False, "missing", None

    try:
        pf = pq.ParquetFile(
            shard
        )

        column_names = set(
            pf.schema.names
        )

        row_count = int(
            pf.metadata.num_rows
        )
    except Exception as e:
        return False, f"metadata_error:{type(e).__name__}", None

    if row_count != expected_units:
        return (
            False,
            f"row_count:{row_count}!={expected_units}",
            None,
        )

    required_identity = {
        "unit_uid",
        "landfire_version_key",
        "terrain_block_id",
    }

    if not required_identity.issubset(
        column_names
    ):
        return False, "missing_identity_columns", None

    missing = sorted(
        FROZEN_V5_FEATURE_SET
        - column_names
    )

    unexpected = sorted(
        c
        for c in column_names
        if c.startswith("landfire_")
        and c not in FROZEN_V5_FEATURE_SET
    )

    if missing:
        return False, f"missing_features:{len(missing)}", None

    if unexpected:
        return False, f"unexpected_features:{len(unexpected)}", None

    try:
        identity = pd.read_parquet(
            shard,
            columns=[
                "unit_uid",
                "landfire_version_key",
                "terrain_block_id",
            ],
        )
    except Exception as e:
        return False, f"identity_read_error:{type(e).__name__}", None

    if not identity["unit_uid"].is_unique:
        return False, "unit_uid_not_unique", None

    expected_uid_set = set(
        production_base.loc[
            (production_base["landfire_version_key"] == version_key)
            & (production_base["terrain_block_id"] == block_id),
            "unit_uid",
        ].astype(str)
    )

    observed_uid_set = set(
        identity["unit_uid"].astype(str)
    )

    if observed_uid_set != expected_uid_set:
        return False, "unit_uid_set_mismatch", None

    if not (
        identity["landfire_version_key"]
        .astype(str)
        .eq(str(version_key))
        .all()
    ):
        return False, "version_mismatch", None

    if not (
        identity["terrain_block_id"]
        .astype(str)
        .eq(str(block_id))
        .all()
    ):
        return False, "block_id_mismatch", None

    summary = {
        "version_key": version_key,
        "block_id": block_id,
        "units": expected_units,
        "feature_rows": row_count,
        "feature_columns": len(column_names),
        "missing_v5_feature_count": 0,
        "unexpected_landfire_feature_count": 0,
        "shard_path": str(shard),
        "resume_status": "validated_existing_shard",
    }

    return True, "valid", summary



## 16C. Incremental QA checkpoint helper

After every completed/failed block, currently accumulated status and QA tables
are written to disk.

This means a kernel interruption does not discard the audit trail already
collected in that session. Feature shards remain the authoritative restart
checkpoint.


In [ ]:

def checkpoint_production_logs(
    block_status_rows,
    tile_qa_frames,
    product_center_qa_frames,
    canopy_alignment_qa_frames,
):
    if block_status_rows:
        pd.DataFrame(
            block_status_rows
        ).to_csv(
            OUTPUT_ROOT / "PRODUCTION_BLOCK_STATUS_V3.csv",
            index=False,
        )

    if tile_qa_frames:
        pd.concat(
            tile_qa_frames,
            ignore_index=True,
        ).to_csv(
            OUTPUT_ROOT / "PRODUCTION_TILE_QA_INCREMENTAL_V3.csv",
            index=False,
        )

    if product_center_qa_frames:
        pd.concat(
            product_center_qa_frames,
            ignore_index=True,
        ).to_csv(
            OUTPUT_ROOT / "PRODUCTION_PRODUCT_CENTER_QA_INCREMENTAL_V3.csv",
            index=False,
        )

    if canopy_alignment_qa_frames:
        pd.concat(
            canopy_alignment_qa_frames,
            ignore_index=True,
        ).to_csv(
            OUTPUT_ROOT / "PRODUCTION_CANOPY_ALIGNMENT_QA_INCREMENTAL_V3.csv",
            index=False,
        )


## Recovery preflight

This scans all expected shards using Parquet metadata + three identity columns.
It does **not** load all feature columns.

The `needs_recompute` rows are the only blocks that the production loop should
actually calculate again.


In [ ]:
preflight_rows = []

for _, row in tqdm(
    block_manifest.iterrows(),
    total=len(block_manifest),
    desc="Recovery preflight",
):
    valid, reason, summary = validate_existing_shard(
        row
    )

    preflight_rows.append({
        "version_key": row["landfire_version_key"],
        "block_id": row["terrain_block_id"],
        "units": int(row["units"]),
        "valid_existing_shard": bool(valid),
        "reason": reason,
        "needs_recompute": not bool(valid),
    })

recovery_preflight = pd.DataFrame(
    preflight_rows
)

display(
    recovery_preflight.groupby(
        [
            "version_key",
            "needs_recompute",
            "reason",
        ],
        as_index=False,
    ).agg(
        blocks=("block_id", "size"),
        units=("units", "sum"),
    )
)

print()
print(
    "Valid existing blocks:",
    f"{int(recovery_preflight['valid_existing_shard'].sum()):,}",
    "/",
    f"{len(recovery_preflight):,}",
)

print(
    "Blocks that will be recomputed:",
    f"{int(recovery_preflight['needs_recompute'].sum()):,}",
)

recovery_preflight.to_csv(
    OUTPUT_ROOT / "RECOVERY_PREFLIGHT_V3.csv",
    index=False,
)


## 17. Execute / resume full resolved-vintage production

In [ ]:

results = []
tile_qa_frames = []
product_center_qa_frames = []
canopy_alignment_qa_frames = []
block_status_rows = []

started_utc = datetime.now(timezone.utc)

for block_number, (_, row) in enumerate(
    block_manifest.iterrows(),
    start=1,
):
    version_key = row["landfire_version_key"]
    block_id = row["terrain_block_id"]

    print()
    print(
        f"[{block_number:,}/{len(block_manifest):,}] "
        f"{version_key} / {block_id} "
        f"({int(row['units']):,} units)"
    )

    # --------------------------------------------------------------
    # Resume path.
    # --------------------------------------------------------------
    if RESUME_COMPLETED_BLOCKS:
        valid, reason, resume_summary = validate_existing_shard(
            row
        )

        if valid:
            results.append(resume_summary)

            block_status_rows.append({
                "version_key": version_key,
                "block_id": block_id,
                "status": "skipped_valid_existing",
                "attempts": 0,
                "error_type": None,
                "error_message": None,
                "finished_utc": datetime.now(timezone.utc).isoformat(),
            })

            print("  SKIP: validated existing shard")

            checkpoint_production_logs(
                block_status_rows,
                tile_qa_frames,
                product_center_qa_frames,
                canopy_alignment_qa_frames,
            )
            continue

        elif reason != "missing":
            print(
                "  Existing shard invalid; recomputing:",
                reason,
            )

    # --------------------------------------------------------------
    # Compute with retry.
    # --------------------------------------------------------------
    last_error = None
    success = False

    for attempt in range(
        1,
        MAX_BLOCK_ATTEMPTS + 1,
    ):
        try:
            result = run_production_block(
                row
            )

            tile_q = result.pop(
                "tile_qa"
            ).copy()

            tile_q[
                "production_attempt"
            ] = attempt

            tile_qa_frames.append(
                tile_q
            )

            pcq = result.pop(
                "product_center_qa"
            ).copy()

            pcq[
                "production_attempt"
            ] = attempt

            product_center_qa_frames.append(
                pcq
            )

            caq = result.pop(
                "canopy_alignment_qa"
            ).copy()

            caq[
                "production_attempt"
            ] = attempt

            canopy_alignment_qa_frames.append(
                caq
            )

            # Normalize the computed-result representation to the same
            # count fields used by validated/resumed shards.
            result[
                "missing_v5_feature_count"
            ] = len(
                result.pop(
                    "missing_v5_features",
                    [],
                )
            )

            result[
                "unexpected_landfire_feature_count"
            ] = len(
                result.pop(
                    "unexpected_landfire_features",
                    [],
                )
            )

            # Free the large in-memory per-unit table from result before
            # retaining only block summary metadata.
            result.pop(
                "features",
                None,
            )

            result[
                "resume_status"
            ] = "computed"

            results.append(
                result
            )

            block_status_rows.append({
                "version_key": version_key,
                "block_id": block_id,
                "status": "completed",
                "attempts": attempt,
                "error_type": None,
                "error_message": None,
                "finished_utc": datetime.now(timezone.utc).isoformat(),
            })

            success = True
            print("  PASS")
            break

        except Exception as e:
            last_error = e

            print(
                f"  attempt {attempt}/{MAX_BLOCK_ATTEMPTS} failed:",
                type(e).__name__,
                str(e)[:500],
            )

            if attempt < MAX_BLOCK_ATTEMPTS:
                time.sleep(
                    RETRY_BACKOFF_S
                    * attempt
                )

    if not success:
        failure_trace = "".join(
            traceback.format_exception(
                type(last_error),
                last_error,
                last_error.__traceback__,
            )
        )

        failure_trace_path = (
            OUTPUT_ROOT /
            f"FAILED_TRACE__{version_key}__{safe_token(block_id)}.txt"
        )

        failure_trace_path.write_text(
            failure_trace,
            encoding="utf-8",
        )

        block_status_rows.append({
            "version_key": version_key,
            "block_id": block_id,
            "status": "failed",
            "attempts": MAX_BLOCK_ATTEMPTS,
            "error_type": type(last_error).__name__,
            "error_message": str(last_error)[:4000],
            "trace_path": str(failure_trace_path),
            "finished_utc": datetime.now(timezone.utc).isoformat(),
        })

        if not CONTINUE_ON_BLOCK_ERROR:
            raise last_error

    # Checkpoint status + accumulated QA after every block.
    checkpoint_production_logs(
        block_status_rows,
        tile_qa_frames,
        product_center_qa_frames,
        canopy_alignment_qa_frames,
    )


block_status = pd.DataFrame(
    block_status_rows
)

production_summary = pd.DataFrame([
    {
        "version_key": r["version_key"],
        "block_id": r["block_id"],
        "units": r["units"],
        "line_samples": r.get("line_samples", np.nan),
        "context_requests": r.get("context_requests", np.nan),
        "feature_rows": r["feature_rows"],
        "feature_columns": r["feature_columns"],
        "missing_v5_feature_count": r.get(
            "missing_v5_feature_count",
            len(r.get("missing_v5_features", [])),
        ),
        "unexpected_landfire_feature_count": r.get(
            "unexpected_landfire_feature_count",
            len(r.get("unexpected_landfire_features", [])),
        ),
        "resume_status": r["resume_status"],
        "shard_path": r["shard_path"],
    }
    for r in results
])

tile_qa = (
    pd.concat(
        tile_qa_frames,
        ignore_index=True,
    )
    if tile_qa_frames
    else pd.DataFrame()
)

product_center_qa = (
    pd.concat(
        product_center_qa_frames,
        ignore_index=True,
    )
    if product_center_qa_frames
    else pd.DataFrame()
)

canopy_alignment_qa = (
    pd.concat(
        canopy_alignment_qa_frames,
        ignore_index=True,
    )
    if canopy_alignment_qa_frames
    else pd.DataFrame()
)

display(
    block_status["status"].value_counts(
        dropna=False
    ).rename_axis("status").reset_index(name="blocks")
)

display(
    production_summary.groupby(
        ["version_key", "resume_status"],
        as_index=False,
    ).agg(
        blocks=("block_id", "size"),
        units=("units", "sum"),
    )
)

production_summary.to_csv(
    OUTPUT_ROOT / "PRODUCTION_BLOCK_SUMMARY_V3.csv",
    index=False,
)

block_status.to_csv(
    OUTPUT_ROOT / "PRODUCTION_BLOCK_STATUS_V3.csv",
    index=False,
)

if not tile_qa.empty:
    tile_qa.to_csv(
        OUTPUT_ROOT / "PRODUCTION_TILE_QA_V3.csv",
        index=False,
    )

if not product_center_qa.empty:
    product_center_qa.to_csv(
        OUTPUT_ROOT / "PRODUCTION_PRODUCT_CENTER_QA_V3.csv",
        index=False,
    )

if not canopy_alignment_qa.empty:
    canopy_alignment_qa.to_csv(
        OUTPUT_ROOT / "PRODUCTION_CANOPY_ALIGNMENT_QA_V3.csv",
        index=False,
    )

elapsed_h = (
    datetime.now(timezone.utc)
    - started_utc
).total_seconds() / 3600.0

print()
print(
    "Production loop elapsed:",
    f"{elapsed_h:.2f} h",
)


## 18. Validate every completed shard against frozen V5

In [ ]:

shard_audit_rows = []

for _, row in block_manifest.iterrows():
    valid, reason, summary = validate_existing_shard(
        row
    )

    shard_audit_rows.append({
        "version_key": row["landfire_version_key"],
        "block_id": row["terrain_block_id"],
        "expected_units": int(row["units"]),
        "valid_shard": bool(valid),
        "reason": reason,
        "shard_path": (
            summary["shard_path"]
            if summary is not None
            else str(
                expected_shard_path(
                    row["landfire_version_key"],
                    row["terrain_block_id"],
                )
            )
        ),
    })

shard_audit = pd.DataFrame(
    shard_audit_rows
)

display(
    shard_audit.groupby(
        ["version_key", "valid_shard", "reason"],
        as_index=False,
    ).size()
)

invalid_shards = shard_audit[
    ~shard_audit["valid_shard"]
].copy()

display(
    invalid_shards.head(200)
)

shard_audit.to_csv(
    OUTPUT_ROOT / "PRODUCTION_SHARD_AUDIT_V3.csv",
    index=False,
)

invalid_shards.to_csv(
    OUTPUT_ROOT / "PRODUCTION_INVALID_SHARDS_V3.csv",
    index=False,
)


## 19. Production-wide feature coverage QA

In [ ]:

coverage_rows = []

core_features = feature_dictionary.loc[
    feature_dictionary["tier"] == "core",
    "feature",
].tolist()

for _, audit_row in tqdm(
    shard_audit[
        shard_audit["valid_shard"]
    ].iterrows(),
    total=int(
        shard_audit["valid_shard"].sum()
    ),
    desc="Coverage audit",
):
    x = pd.read_parquet(
        audit_row["shard_path"],
        columns=[
            "unit_uid",
            "landfire_version_key",
            *core_features,
        ],
    )

    arr = (
        x[core_features]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .to_numpy(float)
    )

    coverage_rows.append({
        "version_key": audit_row["version_key"],
        "block_id": audit_row["block_id"],
        "units": len(x),
        "core_nonmissing_fraction": float(
            np.isfinite(arr).mean()
        ),
    })

production_coverage = pd.DataFrame(
    coverage_rows
)

display(
    production_coverage.groupby(
        "version_key",
        as_index=False,
    ).agg(
        blocks=("block_id", "size"),
        units=("units", "sum"),
        mean_block_core_coverage=(
            "core_nonmissing_fraction",
            "mean",
        ),
        min_block_core_coverage=(
            "core_nonmissing_fraction",
            "min",
        ),
        median_block_core_coverage=(
            "core_nonmissing_fraction",
            "median",
        ),
        max_block_core_coverage=(
            "core_nonmissing_fraction",
            "max",
        ),
    )
)

production_coverage.to_csv(
    OUTPUT_ROOT / "PRODUCTION_FEATURE_COVERAGE_V3.csv",
    index=False,
)


## 20. Full-production final checks

In [ ]:

expected_blocks = len(
    block_manifest
)

expected_units = int(
    block_manifest["units"].sum()
)

valid_blocks = int(
    shard_audit["valid_shard"].sum()
)

valid_units = int(
    shard_audit.loc[
        shard_audit["valid_shard"],
        "expected_units",
    ].sum()
)

failed_blocks = int(
    (
        block_status["status"]
        == "failed"
    ).sum()
) if len(block_status) else 0

checks = pd.DataFrame([
    {
        "check": "all_expected_blocks_have_valid_shards",
        "pass": valid_blocks == expected_blocks,
        "observed": valid_blocks,
        "expected": expected_blocks,
    },
    {
        "check": "all_expected_units_covered",
        "pass": valid_units == expected_units,
        "observed": valid_units,
        "expected": expected_units,
    },
    {
        "check": "no_failed_blocks",
        "pass": failed_blocks == 0,
        "observed": failed_blocks,
        "expected": 0,
    },
    {
        "check": "exact_frozen_v5_schema_all_shards",
        "pass": bool(
            shard_audit["valid_shard"].all()
        ),
        "observed": valid_blocks,
        "expected": expected_blocks,
    },
    {
        "check": "all_newly_computed_context_requests_inside_tiles",
        "pass": bool(
            tile_qa.empty
            or (
                "all_context_requests_inside_tile" in tile_qa.columns
                and tile_qa[
                    "all_context_requests_inside_tile"
                ].all()
            )
        ),
        "observed": (
            int(
                tile_qa[
                    "all_context_requests_inside_tile"
                ].sum()
            )
            if (
                not tile_qa.empty
                and "all_context_requests_inside_tile" in tile_qa.columns
            )
            else 0
        ),
        "expected": (
            len(tile_qa)
            if not tile_qa.empty
            else 0
        ),
    },
])

display(checks)

PRODUCTION_PASS = bool(
    checks["pass"].all()
)

print()
print(
    "04D1E RESOLVED-VINTAGE FULL PRODUCTION:",
    (
        "PASS / READY TO CONSOLIDATE"
        if PRODUCTION_PASS
        else "INCOMPLETE / RERUN FAILED OR MISSING BLOCKS"
    ),
)

checks.to_csv(
    OUTPUT_ROOT / "PRODUCTION_FINAL_CHECKS_V3.csv",
    index=False,
)


## 21. Full-production report

In [ ]:

def tomd(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)


status_summary = (
    block_status.groupby(
        ["version_key", "status"],
        as_index=False,
    )
    .size()
    if len(block_status)
    else pd.DataFrame()
)

coverage_summary = (
    production_coverage.groupby(
        "version_key",
        as_index=False,
    ).agg(
        blocks=("block_id", "size"),
        units=("units", "sum"),
        mean_block_core_coverage=(
            "core_nonmissing_fraction",
            "mean",
        ),
        min_block_core_coverage=(
            "core_nonmissing_fraction",
            "min",
        ),
        median_block_core_coverage=(
            "core_nonmissing_fraction",
            "median",
        ),
        max_block_core_coverage=(
            "core_nonmissing_fraction",
            "max",
        ),
    )
    if len(production_coverage)
    else pd.DataFrame()
)


report = f"""# 04D1E_v2 LANDFIRE Recovery / Resume

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Expected production universe

Blocks: {len(block_manifest):,}

Units: {int(block_manifest['units'].sum()):,}

Versions: {", ".join(PRODUCTION_VERSIONS)}

## Block status

{tomd(status_summary)}

## Shard audit

Valid blocks: {int(shard_audit['valid_shard'].sum()):,} / {len(shard_audit):,}

Invalid blocks: {int((~shard_audit['valid_shard']).sum()):,}

{tomd(invalid_shards.head(100)) if len(invalid_shards) else "No invalid shards."}

## Core feature coverage by vintage

{tomd(coverage_summary)}

## Final checks

{tomd(checks)}

## Decision

`{"PASS / READY TO CONSOLIDATE" if PRODUCTION_PASS else "INCOMPLETE / RERUN REQUIRED"}`

The extraction stage preserves valid structural zero and intentional
positive-conditioned canopy missingness. No model-stage imputation is performed
here.

LF2019L and LF2020 remain outside this production artifact until their archived
raw sources are acquired.
"""

REPORT_PATH = (
    OUTPUT_ROOT /
    "LANDFIRE_RESOLVED_VINTAGE_FULL_PRODUCTION_REPORT_V3.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(report)
print()
print("Saved:", REPORT_PATH)


## 22. Output inventory

In [ ]:

print("=" * 100)
print("04D1E OUTPUTS")
print("=" * 100)

for p in sorted(
    OUTPUT_ROOT.glob("*")
):
    if p.is_file():
        print(
            f"{p.name:<90}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

shards = sorted(
    SHARD_ROOT.glob("*.parquet")
)

print()
print(
    "Feature shards:",
    f"{len(shards):,}",
)

total_shard_gb = sum(
    p.stat().st_size
    for p in shards
) / (1024**3)

print(
    "Feature-shard size:",
    f"{total_shard_gb:.3f} GB",
)

print()
print("DONE:", NOTEBOOK_BUILD_ID)
